# **Collected Data**

In [1]:
import os
import tarfile
import gzip
import shutil
import pandas as pd
import pickle


path = "/home/uynk/Belgeler/Analyze_Projects/AliBaba_GenAI_Dataset/v2026_GenAI"

# Bütün verilerimizi isimleriyle saklayacağımız sözlük
dataframes = {}

def is_gzip(file_path):
    """Bir dosyanın gerçekten gzip formatında olup olmadığını kontrol eder."""
    try:
        with open(file_path, "rb") as f:
            return f.read(2) == b"\x1f\x8b"
    except:
        return False

for file_name in os.listdir(path):
    if file_name.endswith(".tar.gz"):
        file_path = os.path.join(path, file_name)
        
        try:
            # 1. SENARYO: Gerçekten bir tar.gz arşivi mi?
            with tarfile.open(file_path, "r:gz") as tar:
                tar.extractall(path=path)
                extracted_files = tar.getnames()
                
                for extracted in extracted_files:
                    full_path = os.path.join(path, extracted)
                    if not os.path.isfile(full_path): 
                        continue

                    # Eğer tar içinden çıkan dosya da gzip ise
                    if is_gzip(full_path):
                        decompressed_path = full_path + ".csv"
                        print(f"📦 İçerikte Gzip tespit edildi: {extracted} -> Çıkartılıyor...")
                        try:
                            with gzip.open(full_path, "rb") as f_in:
                                with open(decompressed_path, "wb") as f_out:
                                    shutil.copyfileobj(f_in, f_out)
                            full_path = decompressed_path
                        except Exception as e:
                            print(f"❌ Gunzip hatası ({extracted}): {e}")
                            continue

                    # Çıkartılan veriyi DataFrame'e çevir ve kaydet
                    try:
                        df = pd.read_csv(full_path)
                        if df.empty:
                            print(f"⚠️ UYARI: {os.path.basename(full_path)} BOŞ!")
                        else:
                            df_name = os.path.basename(full_path)
                            dataframes[df_name] = df
                            print(f"✅ TAMAM (Tar İçinden): {df_name} başarıyla sözlüğe eklendi.")
                    except Exception as e:
                        print(f"❌ HATA: {os.path.basename(full_path)} okunamadı: {e}")
                        
        except tarfile.ReadError:
            print(f"🔄 BİLGİ: {file_name} bir tar arşivi değil. Doğrudan pandas ile okunuyor...")   # gzip şeklinde gözüken ama aslında direkt csv dosyası şeklinde yayınlanmış veriler için bu kod.
            try:
                # Dosyayı senin verdiğin örnekteki gibi doğrudan okuyoruz
                df = pd.read_csv(file_path, compression='gzip')
                
                if df.empty:
                    print(f"⚠️ UYARI: {file_name} BOŞ!")
                else:
                    dataframes[file_name] = df
                    print(f"✅ TAMAM (Doğrudan): {file_name} okunup sözlüğe eklendi.")
                    
            except Exception as e:
                print(f"💥 KRİTİK HATA: {file_name} ne tar olarak ne de doğrudan okunabildi: {e}")
                
        except Exception as e:
            print(f"💥 BEKLENMEYEN HATA: {file_name} işlenirken bir sorun oluştu: {e}")

print(f"\n--- Tüm işlemler tamamlandı! Toplam {len(dataframes)} adet veri DataFrame'e çevrildi. ---") 

✅ TAMAM (Tar İçinden): pipeline_inference_data_anon.csv başarıyla sözlüğe eklendi.
❌ HATA: ._data_trace_processed.csv okunamadı: 'utf-8' codec can't decode byte 0x90 in position 211: invalid start byte
✅ TAMAM (Tar İçinden): data_trace_processed.csv başarıyla sözlüğe eklendi.
✅ TAMAM (Tar İçinden): queue_size_raw_anon.csv başarıyla sözlüğe eklendi.
✅ TAMAM (Tar İçinden): controlnet_latency_data_anon.csv başarıyla sözlüğe eklendi.
✅ TAMAM (Tar İçinden): lora_request_trace.csv başarıyla sözlüğe eklendi.
✅ TAMAM (Tar İçinden): lora_update_latency_anon.csv başarıyla sözlüğe eklendi.
✅ TAMAM (Tar İçinden): pod_memory_util_anon.csv başarıyla sözlüğe eklendi.
✅ TAMAM (Tar İçinden): model_predict_data_anon.csv başarıyla sözlüğe eklendi.
✅ TAMAM (Tar İçinden): queue_rt_raw_anon.csv başarıyla sözlüğe eklendi.
✅ TAMAM (Tar İçinden): pod_gpu_duty_cycle_anon.csv başarıyla sözlüğe eklendi.
✅ TAMAM (Tar İçinden): pipeline_update_latency_anon.csv başarıyla sözlüğe eklendi.
✅ TAMAM (Tar İçinden): qps.c

# Data Preprocessing

In [2]:
import pandas as pd

def new_feature(df_dict):
    for var in df_dict:
        if "timestamp_anon" in df_dict[var].columns:
            
            df_dict[var]["real_time"] = pd.to_datetime(df_dict[var]["timestamp_anon"], unit="s")
            df_dict[var]["real_time_CST"] = df_dict[var]["real_time"] + pd.Timedelta(hours=8)
            
            df_dict[var].sort_values("real_time", inplace=True)
            df_dict[var].reset_index(drop=True, inplace=True)
        
            print(f"{var}: Created 'real_time' and 'real_time_CST', and sorted data.")
        
        else:
            print(f"{var}: Column 'timestamp_anon' not found")

new_feature(dataframes)


path2 = "/home/uynk/Belgeler/Analyze_Projects/AliBaba_GenAI_Dataset"
os.makedirs(path2, exist_ok=True)

dosya_yolu = f'{path2}/dataframes.pkl'
with open(dosya_yolu, 'wb') as dosya:
    pickle.dump(dataframes, dosya)

print(f"'{dosya_yolu}' adresine başarıyla kaydedildi.")

pipeline_inference_data_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
data_trace_processed.csv: Column 'timestamp_anon' not found
queue_size_raw_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
controlnet_latency_data_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
lora_request_trace.csv: Column 'timestamp_anon' not found
lora_update_latency_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
pod_memory_util_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
model_predict_data_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
queue_rt_raw_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
pod_gpu_duty_cycle_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
pipeline_update_latency_anon.csv: Created 'real_time' and 'real_time_CST', and sorted data.
qps.csv: Created 'real_time' and 'real_time_CST', and sorted data.
pod_gpu_memory_used_bytes_ano